We'll cleaned the posts data file first

Load the dataset

In [3]:
import pandas as pd

reddit_comments_path = r"C:\Users\USER\Desktop\Projects\philosophy_and_popculture\data\pop_texts\reddit\reddit_keyword_with_comments.csv"
df_reddit_comments = pd.read_csv(reddit_comments_path)

df_reddit_comments.head()

,source,subreddit,keyword,post_id,title,text
0,reddit,philosophy,freedom,836a0f,Researcher teaches philosophy to inmates at pr...,Researcher teaches philosophy to inmates at pr...
1,reddit,philosophy,freedom,qxj5a8,Being an employee is a threat to your liberty....,Being an employee is a threat to your liberty....
2,reddit,philosophy,freedom,87jbvl,Facebook has compromised our privacy. But that...,Facebook has compromised our privacy. But that...
3,reddit,philosophy,freedom,yn03or,Yale Professor of Philosophy Jason Stanley arg...,Yale Professor of Philosophy Jason Stanley arg...
4,reddit,philosophy,freedom,p6q0lr,"Freedom is essential for creativity, and to sa...","Freedom is essential for creativity, and to sa..."


In [4]:
df_reddit_comments.columns

Index(['source', 'subreddit', 'keyword', 'post_id', 'title', 'text'], dtype='object')

Rename the column titles

In [5]:
df_reddit_cleaned = df_reddit_comments[['subreddit', 'keyword', 'post_id', 'title', 'text']].copy()

df_reddit_cleaned = df_reddit_cleaned.rename(columns={
    'subreddit': 'subreddit',
    'keyword': 'keyword',
    'post_id': 'post_id',
    'title': 'post_title',
    'text': 'post_body'
})

df_reddit_cleaned.head()

,subreddit,keyword,post_id,post_title,post_body
0,philosophy,freedom,836a0f,Researcher teaches philosophy to inmates at pr...,Researcher teaches philosophy to inmates at pr...
1,philosophy,freedom,qxj5a8,Being an employee is a threat to your liberty....,Being an employee is a threat to your liberty....
2,philosophy,freedom,87jbvl,Facebook has compromised our privacy. But that...,Facebook has compromised our privacy. But that...
3,philosophy,freedom,yn03or,Yale Professor of Philosophy Jason Stanley arg...,Yale Professor of Philosophy Jason Stanley arg...
4,philosophy,freedom,p6q0lr,"Freedom is essential for creativity, and to sa...","Freedom is essential for creativity, and to sa..."


We'll lowercase everything, remove extra whitespaces

In [7]:
import re

def clean_reddit_text(text):
    if pd.isnull(text):
        return ""
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)  # collapse multiple spaces/newlines
    text = text.strip()
    return text

# Keep originals
df_reddit_cleaned["post_title_original"] = df_reddit_cleaned["post_title"]
df_reddit_cleaned["post_body_original"] = df_reddit_cleaned["post_body"]

# Cleaned
df_reddit_cleaned["post_title"] = df_reddit_cleaned["post_title"].apply(clean_reddit_text)
df_reddit_cleaned["post_body"] = df_reddit_cleaned["post_body"].apply(clean_reddit_text)

df_reddit_cleaned[["post_title", "post_body"]].head()


,post_title,post_body
0,researcher teaches philosophy to inmates at pr...,researcher teaches philosophy to inmates at pr...
1,being an employee is a threat to your liberty....,being an employee is a threat to your liberty....
2,facebook has compromised our privacy. but that...,facebook has compromised our privacy. but that...
3,yale professor of philosophy jason stanley arg...,yale professor of philosophy jason stanley arg...
4,"freedom is essential for creativity, and to sa...","freedom is essential for creativity, and to sa..."


check for missing and duplicate data

In [9]:
# Check for missing values
print("Missing values:\n", df_reddit_cleaned[["post_title", "post_body"]].isnull().sum())

# Check for empty strings (some cells aren't NaN but are just empty)
print("\nEmpty strings:\n", (df_reddit_cleaned[["post_title", "post_body"]] == "").sum())

# Check for duplicate text posts (based on title + body)
df_reddit_cleaned["is_duplicate"] = df_reddit_cleaned.duplicated(subset=["post_title", "post_body"])

# How many duplicates?
print("\nNumber of duplicates:", df_reddit_cleaned["is_duplicate"].sum())

# Show a few duplicates
df_reddit_cleaned[df_reddit_cleaned["is_duplicate"]].head()


Missing values:
 post_title    0
post_body     0
dtype: int64

Empty strings:
 post_title    0
post_body     0
dtype: int64

Number of duplicates: 70


,subreddit,keyword,post_id,post_title,post_body,post_title_original,post_body_original,is_duplicate
1015,Existentialism,meaning,1iem9ta,"life is meaningless, free will is an illusion,...","life is meaningless, free will is an illusion,...","Life is meaningless, free will is an illusion,...","Life is meaningless, free will is an illusion,...",True
1055,Stoicism,meaning,jp6s5z,everything can be taken from a man but one thi...,everything can be taken from a man but one thi...,Everything can be taken from a man but one thi...,Everything can be taken from a man but one thi...,True
1119,psychology,meaning,t6gad7,having a sense of meaning is less important fo...,having a sense of meaning is less important fo...,Having a sense of meaning is less important fo...,Having a sense of meaning is less important fo...,True
1296,Libertarian,meaning,qdgi42,"biden: ""the two things that concern me. one, a...","biden: ""the two things that concern me. one, a...","Biden: ""The two things that concern me. One, a...","Biden: ""The two things that concern me. One, a...",True
1332,Anarchism,meaning,1l6qn06,the freedom flotilla nears gaza; israel determ...,the freedom flotilla nears gaza; israel determ...,The Freedom Flotilla Nears Gaza; Israel Determ...,The Freedom Flotilla Nears Gaza; Israel Determ...,True


drop the duplicates

In [12]:
df_reddit_cleaned = df_reddit_cleaned.drop_duplicates(subset=["post_title", "post_body"]).reset_index(drop=True)

df_reddit_cleaned.shape

(4114, 8)

save the cleaned file

#### Now let's work on the comment files

since there are a lot of files, let's get a good picture of what everything looks like

In [34]:
from pathlib import Path
import pandas as pd

# Set the folder path
reddit_folder = Path("C:/Users/USER/Desktop/Projects/philosophy_and_popculture/data/pop_texts/reddit/reddit_comments")

# Get all Reddit CSV files (sorted to maintain consistent order)
all_csv_files = sorted(reddit_folder.glob("*.csv"))

# Confirm total number of files
print(f"Total Reddit CSV files: {len(all_csv_files)}")

# Load the first file
first_file_path = all_csv_files[0]
print(f"Now working on: {first_file_path.name}")

# Load and display
df_reddit_1 = pd.read_csv(first_file_path)
df_reddit_1.head()

Total Reddit CSV files: 225
Now working on: reddit (1).csv


,flex href,h-full src,truncate,text-neutral-content-weak href,text-neutral-content-weak,py-0,overflow-hidden href 2,py-0 9
0,https://www.reddit.com/user/belltowerbats/,https://www.redditstatic.com/avatars/defaults/...,belltowerbats,https://www.reddit.com/r/socialskills/comments...,6y ago,"I relate to this a lot, although I’m far less ...",[object Object],[X] I'm in this post and I don't like it
1,https://www.reddit.com/user/worstusernameever00/,https://www.redditstatic.com/avatars/defaults/...,worstusernameever00,https://www.reddit.com/r/socialskills/comments...,6y ago,"Yes, I feel like an outsider a lot. Your post ...",NaN,NaN
2,https://www.reddit.com/user/faustian1/,NaN,faustian1,https://www.reddit.com/r/socialskills/comments...,6y ago,"Yes sir, I think you nailed it with that descr...",[object Object],NaN
3,https://www.reddit.com/user/callmekanga/,NaN,[deleted],https://www.reddit.com/r/socialskills/comments...,6y ago,"Yep, I can relate, I once went to a bar and on...",[object Object],"When I'm going out alone, that feeling of bein..."
4,NaN,NaN,[deleted],https://www.reddit.com/r/socialskills/comments...,6y ago,Definitely. It can be a lonely world out there...,[object Object],NaN


rename the columns

In [35]:
df_reddit_1.columns = [
    "user_url",         # Column 0
    "avatar_url",       # Column 1 (can drop later)
    "username",         # Column 2
    "comment_url",      # Column 3
    "time_posted",      # Column 4
    "comment",          # Column 5
    "unknown_1",        # Column 6 (likely placeholder or JS object)
    "reply"             # Column 7 (optional reply)
]

df_reddit_1.head()


,user_url,avatar_url,username,comment_url,time_posted,comment,unknown_1,reply
0,https://www.reddit.com/user/belltowerbats/,https://www.redditstatic.com/avatars/defaults/...,belltowerbats,https://www.reddit.com/r/socialskills/comments...,6y ago,"I relate to this a lot, although I’m far less ...",[object Object],[X] I'm in this post and I don't like it
1,https://www.reddit.com/user/worstusernameever00/,https://www.redditstatic.com/avatars/defaults/...,worstusernameever00,https://www.reddit.com/r/socialskills/comments...,6y ago,"Yes, I feel like an outsider a lot. Your post ...",NaN,NaN
2,https://www.reddit.com/user/faustian1/,NaN,faustian1,https://www.reddit.com/r/socialskills/comments...,6y ago,"Yes sir, I think you nailed it with that descr...",[object Object],NaN
3,https://www.reddit.com/user/callmekanga/,NaN,[deleted],https://www.reddit.com/r/socialskills/comments...,6y ago,"Yep, I can relate, I once went to a bar and on...",[object Object],"When I'm going out alone, that feeling of bein..."
4,NaN,NaN,[deleted],https://www.reddit.com/r/socialskills/comments...,6y ago,Definitely. It can be a lonely world out there...,[object Object],NaN


Drop Unneeded Columns

In [37]:
df_reddit_1 = df_reddit_1.drop(columns=["user_url", "avatar_url", "unknown_1"])
df_reddit_1.head()


,username,comment_url,time_posted,comment,reply
0,belltowerbats,https://www.reddit.com/r/socialskills/comments...,6y ago,"I relate to this a lot, although I’m far less ...",[X] I'm in this post and I don't like it
1,worstusernameever00,https://www.reddit.com/r/socialskills/comments...,6y ago,"Yes, I feel like an outsider a lot. Your post ...",NaN
2,faustian1,https://www.reddit.com/r/socialskills/comments...,6y ago,"Yes sir, I think you nailed it with that descr...",NaN
3,[deleted],https://www.reddit.com/r/socialskills/comments...,6y ago,"Yep, I can relate, I once went to a bar and on...","When I'm going out alone, that feeling of bein..."
4,[deleted],https://www.reddit.com/r/socialskills/comments...,6y ago,Definitely. It can be a lonely world out there...,NaN


convert time_posted to a proper datetime

In [39]:
from datetime import datetime
import re

def parse_relative_time(text):
    now = datetime.now()
    if isinstance(text, str):
        match = re.match(r'(\d+)([a-z]+)', text)
        if match:
            value, unit = int(match.group(1)), match.group(2)
            if unit == 'y':
                return now.replace(year=now.year - value)
            elif unit == 'mo':
                month = now.month - value
                year = now.year
                while month <= 0:
                    month += 12
                    year -= 1
                return now.replace(year=year, month=month)
            elif unit == 'd':
                return now - pd.Timedelta(days=value)
            elif unit == 'h':
                return now - pd.Timedelta(hours=value)
            elif unit == 'm':
                return now - pd.Timedelta(minutes=value)
    return pd.NaT

df_reddit_1["time_posted_parsed"] = df_reddit_1["time_posted"].apply(parse_relative_time)
df_reddit_1.head()


,username,comment_url,time_posted,comment,reply,time_posted_parsed
0,belltowerbats,https://www.reddit.com/r/socialskills/comments...,6y ago,"I relate to this a lot, although I’m far less ...",[X] I'm in this post and I don't like it,2019-07-14 00:11:05.770737
1,worstusernameever00,https://www.reddit.com/r/socialskills/comments...,6y ago,"Yes, I feel like an outsider a lot. Your post ...",NaN,2019-07-14 00:11:05.771048
2,faustian1,https://www.reddit.com/r/socialskills/comments...,6y ago,"Yes sir, I think you nailed it with that descr...",NaN,2019-07-14 00:11:05.771060
3,[deleted],https://www.reddit.com/r/socialskills/comments...,6y ago,"Yep, I can relate, I once went to a bar and on...","When I'm going out alone, that feeling of bein...",2019-07-14 00:11:05.771067
4,[deleted],https://www.reddit.com/r/socialskills/comments...,6y ago,Definitely. It can be a lonely world out there...,NaN,2019-07-14 00:11:05.771073


extract the subreddit name from comment_url

In [41]:
import re

def extract_subreddit(url):
    if isinstance(url, str):
        match = re.search(r'reddit\.com/r/([^/]+)/', url)
        if match:
            return match.group(1)
    return None

df_reddit_1["subreddit"] = df_reddit_1["comment_url"].apply(extract_subreddit)
df_reddit_1.head()


,username,comment_url,time_posted,comment,reply,time_posted_parsed,subreddit
0,belltowerbats,https://www.reddit.com/r/socialskills/comments...,6y ago,"I relate to this a lot, although I’m far less ...",[X] I'm in this post and I don't like it,2019-07-14 00:11:05.770737,socialskills
1,worstusernameever00,https://www.reddit.com/r/socialskills/comments...,6y ago,"Yes, I feel like an outsider a lot. Your post ...",NaN,2019-07-14 00:11:05.771048,socialskills
2,faustian1,https://www.reddit.com/r/socialskills/comments...,6y ago,"Yes sir, I think you nailed it with that descr...",NaN,2019-07-14 00:11:05.771060,socialskills
3,[deleted],https://www.reddit.com/r/socialskills/comments...,6y ago,"Yep, I can relate, I once went to a bar and on...","When I'm going out alone, that feeling of bein...",2019-07-14 00:11:05.771067,socialskills
4,[deleted],https://www.reddit.com/r/socialskills/comments...,6y ago,Definitely. It can be a lonely world out there...,NaN,2019-07-14 00:11:05.771073,socialskills


lowercase the comments

In [43]:
df_reddit_1["comment_original"] = df_reddit_1["comment"]
df_reddit_1["comment"] = df_reddit_1["comment"].str.lower()
df_reddit_1.head()

,username,comment_url,time_posted,comment,reply,time_posted_parsed,subreddit,comment_original
0,belltowerbats,https://www.reddit.com/r/socialskills/comments...,6y ago,"i relate to this a lot, although i’m far less ...",[X] I'm in this post and I don't like it,2019-07-14 00:11:05.770737,socialskills,"I relate to this a lot, although I’m far less ..."
1,worstusernameever00,https://www.reddit.com/r/socialskills/comments...,6y ago,"yes, i feel like an outsider a lot. your post ...",NaN,2019-07-14 00:11:05.771048,socialskills,"Yes, I feel like an outsider a lot. Your post ..."
2,faustian1,https://www.reddit.com/r/socialskills/comments...,6y ago,"yes sir, i think you nailed it with that descr...",NaN,2019-07-14 00:11:05.771060,socialskills,"Yes sir, I think you nailed it with that descr..."
3,[deleted],https://www.reddit.com/r/socialskills/comments...,6y ago,"yep, i can relate, i once went to a bar and on...","When I'm going out alone, that feeling of bein...",2019-07-14 00:11:05.771067,socialskills,"Yep, I can relate, I once went to a bar and on..."
4,[deleted],https://www.reddit.com/r/socialskills/comments...,6y ago,definitely. it can be a lonely world out there...,NaN,2019-07-14 00:11:05.771073,socialskills,Definitely. It can be a lonely world out there...


work on emojis

In [44]:
import re

def remove_emojis(text):
    if not isinstance(text, str):
        return text
    emoji_pattern = re.compile("["
        u"\U0001F600-\U0001F64F"  # emoticons
        u"\U0001F300-\U0001F5FF"  # symbols & pictographs
        u"\U0001F680-\U0001F6FF"  # transport & map symbols
        u"\U0001F1E0-\U0001F1FF"  # flags
        u"\U00002700-\U000027BF"  # dingbats
        u"\U000024C2-\U0001F251"
        "]+", flags=re.UNICODE)
    return emoji_pattern.sub(r'', text)

df_reddit_1["comment"] = df_reddit_1["comment"].apply(remove_emojis)
df_reddit_1.head()

,username,comment_url,time_posted,comment,reply,time_posted_parsed,subreddit,comment_original
0,belltowerbats,https://www.reddit.com/r/socialskills/comments...,6y ago,"i relate to this a lot, although i’m far less ...",[X] I'm in this post and I don't like it,2019-07-14 00:11:05.770737,socialskills,"I relate to this a lot, although I’m far less ..."
1,worstusernameever00,https://www.reddit.com/r/socialskills/comments...,6y ago,"yes, i feel like an outsider a lot. your post ...",NaN,2019-07-14 00:11:05.771048,socialskills,"Yes, I feel like an outsider a lot. Your post ..."
2,faustian1,https://www.reddit.com/r/socialskills/comments...,6y ago,"yes sir, i think you nailed it with that descr...",NaN,2019-07-14 00:11:05.771060,socialskills,"Yes sir, I think you nailed it with that descr..."
3,[deleted],https://www.reddit.com/r/socialskills/comments...,6y ago,"yep, i can relate, i once went to a bar and on...","When I'm going out alone, that feeling of bein...",2019-07-14 00:11:05.771067,socialskills,"Yep, I can relate, I once went to a bar and on..."
4,[deleted],https://www.reddit.com/r/socialskills/comments...,6y ago,definitely. it can be a lonely world out there...,NaN,2019-07-14 00:11:05.771073,socialskills,Definitely. It can be a lonely world out there...


merge comment and reply into a single column

In [45]:
def merge_comment_and_reply(row):
    if pd.isna(row["reply"]):
        return row["comment"]
    return row["comment"] + " " + str(row["reply"])

df_reddit_1["comment"] = df_reddit_1.apply(merge_comment_and_reply, axis=1)
df_reddit_1.drop(columns=["reply"], inplace=True)
df_reddit_1.head()

,username,comment_url,time_posted,comment,time_posted_parsed,subreddit,comment_original
0,belltowerbats,https://www.reddit.com/r/socialskills/comments...,6y ago,"i relate to this a lot, although i’m far less ...",2019-07-14 00:11:05.770737,socialskills,"I relate to this a lot, although I’m far less ..."
1,worstusernameever00,https://www.reddit.com/r/socialskills/comments...,6y ago,"yes, i feel like an outsider a lot. your post ...",2019-07-14 00:11:05.771048,socialskills,"Yes, I feel like an outsider a lot. Your post ..."
2,faustian1,https://www.reddit.com/r/socialskills/comments...,6y ago,"yes sir, i think you nailed it with that descr...",2019-07-14 00:11:05.771060,socialskills,"Yes sir, I think you nailed it with that descr..."
3,[deleted],https://www.reddit.com/r/socialskills/comments...,6y ago,"yep, i can relate, i once went to a bar and on...",2019-07-14 00:11:05.771067,socialskills,"Yep, I can relate, I once went to a bar and on..."
4,[deleted],https://www.reddit.com/r/socialskills/comments...,6y ago,definitely. it can be a lonely world out there...,2019-07-14 00:11:05.771073,socialskills,Definitely. It can be a lonely world out there...


Rearrange the columns

In [46]:
# Reorder columns
df_reddit_1 = df_reddit_1[[
    "username", "subreddit", "comment_url",
    "time_posted", "time_posted_parsed",
    "comment", "comment_original"
]]

# Export
df_reddit_1.to_csv("reddit_cleaned_1.csv", index=False)

df_reddit_1.head()

,username,subreddit,comment_url,time_posted,time_posted_parsed,comment,comment_original
0,belltowerbats,socialskills,https://www.reddit.com/r/socialskills/comments...,6y ago,2019-07-14 00:11:05.770737,"i relate to this a lot, although i’m far less ...","I relate to this a lot, although I’m far less ..."
1,worstusernameever00,socialskills,https://www.reddit.com/r/socialskills/comments...,6y ago,2019-07-14 00:11:05.771048,"yes, i feel like an outsider a lot. your post ...","Yes, I feel like an outsider a lot. Your post ..."
2,faustian1,socialskills,https://www.reddit.com/r/socialskills/comments...,6y ago,2019-07-14 00:11:05.771060,"yes sir, i think you nailed it with that descr...","Yes sir, I think you nailed it with that descr..."
3,[deleted],socialskills,https://www.reddit.com/r/socialskills/comments...,6y ago,2019-07-14 00:11:05.771067,"yep, i can relate, i once went to a bar and on...","Yep, I can relate, I once went to a bar and on..."
4,[deleted],socialskills,https://www.reddit.com/r/socialskills/comments...,6y ago,2019-07-14 00:11:05.771073,definitely. it can be a lonely world out there...,Definitely. It can be a lonely world out there...


In [47]:
# Save cleaned version of the first Reddit file
df_reddit_1.to_csv("reddit_cleaned_1.csv", index=False)

work on the rest of the files in batches

In [62]:
import pandas as pd
import os
import re
from datetime import datetime

# === Configuration ===
reddit_dir = r"C:\Users\USER\Desktop\Projects\philosophy_and_popculture\data\pop_texts\reddit\reddit_comments"
output_file = "cleaned_reddit_sample.csv"
num_files_to_test = 5  # Change this to scale later

# === Cleaning Function ===
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r'<.*?>', '', text)                       # Remove HTML tags
    text = re.sub(r'http\S+', '', text)                     # Remove URLs
    text = re.sub(r'[^\x00-\x7F]+', '', text)               # Remove emojis/symbols
    text = re.sub(r'[^a-zA-Z0-9\s.,!?\'\"@#]', '', text)    # Keep alphanum + basic punctuations
    text = text.lower().strip()                             # Lowercase and strip
    text = re.sub(r'\s+', ' ', text)                        # Remove excess spaces
    return text if len(text) > 10 else ""

# === Column Detection ===
structure_to_column = {
    # Add more mappings if needed
    'reddit (1).csv': 'py-0',
    'reddit (1).xlsx': 'py-0',
    'reddit (10).csv': 'py-0',
    'reddit (11).csv': 'py-0',
    'reddit (12).csv': 'py-0',
}

# === Helper: Try parse timestamp ===
def parse_time(val):
    try:
        return pd.to_datetime(val)
    except:
        return None

# === Process Files ===
cleaned_data = []

files = [f for f in os.listdir(reddit_dir) if f.endswith(('.csv', '.xlsx'))]
files.sort()  # Alphabetical order to make batch reproducible

for file in files[:num_files_to_test]:
    path = os.path.join(reddit_dir, file)
    print(f"Processing: {file}")
    
    try:
        df = pd.read_csv(path) if file.endswith(".csv") else pd.read_excel(path)
    except Exception as e:
        print(f"Error reading {file}: {e}")
        continue

    comment_col = structure_to_column.get(file)
    if comment_col not in df.columns:
        print(f"Comment column not found in {file}")
        continue

    for _, row in df.iterrows():
        comment = str(row.get(comment_col, "")).strip()
        cleaned_comment = clean_text(comment)

        cleaned_data.append({
            "username": row.get("username", ""),
            "subreddit": row.get("subreddit", ""),
            "comment_url": row.get("comment_url", ""),
            "time_posted": row.get("time_posted", ""),
            "time_posted_parsed": parse_time(row.get("time_posted", "")),
            "comment_original": comment,
            "comment": cleaned_comment,
            "file": file
        })

# === Save Output ===
df_out = pd.DataFrame(cleaned_data)
df_out.to_csv(output_file, index=False)
print(f"\n✅ Done! Sample saved to {output_file}")

Processing: comment_column_mapping.csv
Comment column not found in comment_column_mapping.csv
Processing: reddit (1).csv
Processing: reddit (1).xlsx
Processing: reddit (10).csv
Processing: reddit (100).csv
Comment column not found in reddit (100).csv

✅ Done! Sample saved to cleaned_reddit_sample.csv


Scan the reddit files for column structures

In [16]:
import sys
!{sys.executable} -m pip install emoji

  Using cached emoji-2.14.1-py3-none-any.whl.metadata (5.7 kB)
Using cached emoji-2.14.1-py3-none-any.whl (590 kB)


clean all of the files

File listing and their basic info

In [445]:
from pathlib import Path

# Set folder paths
raw_folder = Path(r"C:\Users\USER\Desktop\Projects\philosophy_and_popculture\data\pop_texts\reddit\reddit_comments")
cleaned_folder = Path(r"C:\Users\USER\Desktop\Projects\philosophy_and_popculture\data\pop_texts\reddit\reddit_comments_cleaned")
cleaned_folder.mkdir(parents=True, exist_ok=True)

# List all raw files (CSV and Excel)
raw_files = sorted([f for f in raw_folder.iterdir() if f.suffix in ['.csv', '.xlsx']])

# List already cleaned files (removing the "_cleaned" suffix to match raw names)
cleaned_files = {f.stem.replace('_cleaned', '') for f in cleaned_folder.glob("*_cleaned.csv")}

# Get the next unprocessed file
next_file = None
for file in raw_files:
    if file.stem not in cleaned_files:
        next_file = file
        break

print(f"👉 Next file to clean: {next_file.name}" if next_file else "✅ All files cleaned.")


✅ All files cleaned.


In [446]:
import pandas as pd

# Load the detected file with proper format
input_path = next_file
print("📂 Currently working on file:", input_path.name)

if input_path.suffix == '.csv':
    df = pd.read_csv(input_path, dtype=str, encoding='utf-8', engine='python')
elif input_path.suffix == '.xlsx':
    df = pd.read_excel(input_path, dtype=str)
else:
    raise ValueError(f"Unsupported file format: {input_path.suffix}")

# Print structure
print("🧾 Columns:", df.columns.tolist())
df.head()


AttributeError: 'NoneType' object has no attribute 'name'

In [443]:

# 🧹 Step 2a: Rename Columns
column_map = {
    'truncate': 'comment_author',
    'text-neutral-content-weak': 'comment_time',
    'text-neutral-content-weak href': 'comment_url',
    'flex href': 'user_url',
    'h-full src': 'user_avatar',
    'overflow-hidden href': 'unknown_meta',
    'overflow-hidden href 2': 'unknown_meta_2'
}

existing_cols = [col for col in column_map if col in df.columns]
df = df[existing_cols + [col for col in df.columns if col.startswith('py-0')]]
df = df.rename(columns={col: column_map[col] for col in existing_cols})

# 🧹 Step 2b: Unpivot Comment Fragments into Rows
comment_columns = [col for col in df.columns if col.startswith('py-0')]

df_long = df.melt(
    id_vars=[col for col in df.columns if col not in comment_columns],
    value_vars=comment_columns,
    var_name='comment_fragment_column',
    value_name='comment_body'
)
df_long = df_long.dropna(subset=['comment_body'])


In [444]:
# ✅ Step 2b – Build cleaned_df correctly with fallback chaining (NO Series ambiguity)

# 📂 File: reddit - 2025-07-13T174253.245.csv

# Safely select comment fields with fallback chaining
comment_original = df_long.get('comment_body_2')
if comment_original is None:
    comment_original = df_long.get('comment_body_1')

comment_final = df_long.get('cleaned_comment_body')
if comment_final is None:
    comment_final = comment_original

# Fallback to comment_body if none of those exist
if comment_final is None:
    comment_final = df_long.get('comment_body')
if comment_original is None:
    comment_original = df_long.get('comment_body')



# 🧼 Build standardized structure
cleaned_df = pd.DataFrame({
    'username': df_long.get('comment_author'),
    'subreddit': df_long['comment_url'].str.extract(r'/r/([^/]+)/', expand=False) if 'comment_url' in df_long else None,
    'comment_url': df_long.get('comment_url'),
    'time_posted': '',  # Not present in this file
    'time_posted_parsed': '',
    'comment': comment_final,
    'comment_original': comment_original,
    'post_title': None,
    'post_body': None,
})

# 📑 Step 3 – Enforce final column order
expected_order = [
    'username',
    'subreddit',
    'comment_url',
    'time_posted',
    'time_posted_parsed',
    'comment',
    'comment_original',
    'post_title',
    'post_body'
]
cleaned_df = cleaned_df[expected_order]

# 📤 Preview output
print("📂 Currently working on:", input_path.name)
print("🧾 Final cleaned_df preview:")
print(cleaned_df.head())

# 💾 Step 4 – Save cleaned output
output_path = cleaned_folder / f"{input_path.stem}_cleaned.csv"
cleaned_df.to_csv(output_path, index=False)
print(f"✅ Cleaned and saved: {output_path.name}")


📂 Currently working on: reddit6).xlsx
🧾 Final cleaned_df preview:
         username       subreddit  \
0      zuttozutto  AskWomenOver30   
1       cnote4711  AskWomenOver30   
2       [deleted]  AskWomenOver30   
3  throwoheiusfnk  AskWomenOver30   
4       litetears  AskWomenOver30   

                                         comment_url time_posted  \
0  https://www.reddit.com/r/AskWomenOver30/commen...               
1  https://www.reddit.com/r/AskWomenOver30/commen...               
2  https://www.reddit.com/r/AskWomenOver30/commen...               
3  https://www.reddit.com/r/AskWomenOver30/commen...               
4  https://www.reddit.com/r/AskWomenOver30/commen...               

  time_posted_parsed                                            comment  \
0                     I don't know if my experience is the same, but...   
1                     I was stuck in a similar rut and something tha...   
2                     I’m single and live alone and I sometimes fall...   
3 

##### We've manually cleaned all of the reddit comment data, now let's merge them into one file

In [1]:
from pathlib import Path
import pandas as pd

# Define the cleaned folder path
cleaned_folder = Path(r"C:\Users\USER\Desktop\Projects\philosophy_and_popculture\data\pop_texts\reddit\reddit_comments_cleaned")
merged_file = cleaned_folder / "reddit_comments_merged.csv"

# List all CSV files in the cleaned folder (excluding the merged file itself if re-run)
csv_files = [f for f in cleaned_folder.glob("*.csv") if f.name != merged_file.name]

# Confirm how many files we're merging
print(f"📁 Total cleaned files to merge: {len(csv_files)}")

# Optional: Display a few file names
for f in csv_files[:5]:
    print("🔹", f.name)

# Load and concatenate all DataFrames
df_list = []
for file in csv_files:
    try:
        df = pd.read_csv(file)
        df['source_file'] = file.name  # Optional: track origin
        df_list.append(df)
    except Exception as e:
        print(f"⚠️ Error reading {file.name}: {e}")

# Concatenate all into a single DataFrame
merged_df = pd.concat(df_list, ignore_index=True)

# Final shape check
print("✅ Merged shape:", merged_df.shape)
print("🧾 Columns:", list(merged_df.columns))

# Save to file
merged_df.to_csv(merged_file, index=False)
print(f"💾 Merged file saved to: {merged_file}")


📁 Total cleaned files to merge: 226
🔹 reddit (1).csv
🔹 reddit (1)_cleaned.csv
🔹 reddit (10)_cleaned.csv
🔹 reddit (100)_cleaned.csv
🔹 reddit (11)_cleaned.csv
✅ Merged shape: (29972, 12)
🧾 Columns: ['username', 'subreddit', 'comment_url', 'time_posted', 'time_posted_parsed', 'comment', 'comment_original', 'post_title', 'post_body', 'data_type', 'source_file', 'comment_author']
💾 Merged file saved to: C:\Users\USER\Desktop\Projects\philosophy_and_popculture\data\pop_texts\reddit\reddit_comments_cleaned\reddit_comments_merged.csv


##### let's take a look at the other reddit data

In [1]:
# Load and inspect the cleaned Reddit posts file
import pandas as pd
from pathlib import Path

posts_file = Path(r"C:\Users\USER\Desktop\Projects\philosophy_and_popculture\data\pop_texts\reddit\cleaned_reddit_posts.csv")

# Load with low_memory=False in case of mixed types
reddit_posts_df = pd.read_csv(posts_file, low_memory=False)

# Show shape and columns
print("📄 File shape:", reddit_posts_df.shape)
print("🧾 Columns:", list(reddit_posts_df.columns))

# Preview a few rows
reddit_posts_df.head(3)


📄 File shape: (4114, 8)
🧾 Columns: ['subreddit', 'keyword', 'post_id', 'post_title', 'post_body', 'post_title_original', 'post_body_original', 'is_duplicate']


,subreddit,keyword,post_id,post_title,post_body,post_title_original,post_body_original,is_duplicate
0,philosophy,freedom,836a0f,researcher teaches philosophy to inmates at pr...,researcher teaches philosophy to inmates at pr...,Researcher teaches philosophy to inmates at pr...,Researcher teaches philosophy to inmates at pr...,False
1,philosophy,freedom,qxj5a8,being an employee is a threat to your liberty....,being an employee is a threat to your liberty....,Being an employee is a threat to your liberty....,Being an employee is a threat to your liberty....,False
2,philosophy,freedom,87jbvl,facebook has compromised our privacy. but that...,facebook has compromised our privacy. but that...,Facebook has compromised our privacy. But that...,Facebook has compromised our privacy. But that...,False


##### let's do a safe fuzzy join between the reddit comments and posts based on matching

In [33]:
merged_df['post_found'] = merged_df['post_title_post'].notna()
